# Experimentos SciBERT/SPECTER con datos limpios (issue #104)

Orquestador **reanudable** para correr de noche. La lógica vive en `clasificacion_funcion.{seleccion_datos,
experimento_104, registro_mlflow}`. Todo se registra en MLflow (`clasificacion-funcion-104`).

> **Provisional**: no hay Test Gold humano. Las métricas son contra `val` (V fija), `dev_haiku` y `ref_gpt`.
> **Regla anti-fuga**: se excluyen los citantes de 2014 y 2015, el manifiesto v2, TG2700, taller y piloto.

Primero `DRY = True` (64 casos, 1 época) para validar el flujo; luego `DRY = False`.

In [ ]:
import json
import sys
import time
import traceback
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO))  # `src.tracking` y `proyecto_de_grado` (namespace desde la raiz)

import mlflow

from clasificacion_funcion import adaptador_proyecto_grado as ad
from clasificacion_funcion import clasificador_torch as ct
from clasificacion_funcion import experimento_104 as ex
from clasificacion_funcion import registro_mlflow as rm
from clasificacion_funcion import seleccion_datos as sd

In [ ]:
DRY = True          # True: 64 casos y 1 epoca por run, experimento y carpetas aparte
HORAS_MAX = 7.0     # se omiten las fases restantes al superarlo
SEMILLAS = (42, 1234, 2026)

SUFIJO = "-dry" if DRY else ""
ARTIFACTS = REPO / f"proyecto_de_grado/artifacts/experimentos_104{'_dry' if DRY else ''}"
EVIDENCIAS = REPO / f"proyecto_de_grado/docs/evidencias_experimentos_104{'_dry' if DRY else ''}"
CORRIDAS = ARTIFACTS / "corridas"
T0 = time.time()
FALLAS = []

rm.configurar(rm.EXPERIMENTO + SUFIJO)
print("tracking:", mlflow.get_tracking_uri(), "| DRY =", DRY)


def hay_tiempo() -> bool:
    return (time.time() - T0) / 3600 < HORAS_MAX


def correr(nombre, fn, *a, **kw):
    """Ejecuta un run sin detener el notebook: la falla queda FAILED en MLflow y en FALLAS."""
    if not hay_tiempo():
        print(f"[omitido por HORAS_MAX] {nombre}")
        return None
    t = time.time()
    try:
        run_id = fn(nombre, *a, **kw)
        print(f"[ok {time.time() - t:6.0f}s] {nombre} -> {run_id}")
        return run_id
    except Exception:
        FALLAS.append(nombre)
        print(f"[FALLO] {nombre}\n{traceback.format_exc()}")
        return None


def puntaje(run_id):
    """Macro-F1 medio en dev_haiku y ref_gpt (criterio de eleccion, registrado en MLflow)."""
    if not run_id:
        return -1.0
    m = mlflow.get_run(run_id).data.metrics
    return sum(m.get(k, 0.0) for k in ("dev_haiku_f1_macro", "ref_gpt_f1_macro")) / 2

## Fase 0 — Datos limpios (P, V, D1–D6), asserts de fuga y auditoría de fuentes

In [ ]:
FUENTES = ad.cargar_fuentes_104()
PAPERS = ad.cargar_papers()
RES = sd.construir_todo(FUENTES)          # lanza ValueError si hay cualquier fuga
V, VARIANTES = RES["v"], RES["variantes"]
MANIFIESTO = sd.escribir_artefactos(RES, ARTIFACTS / "datos", EVIDENCIAS, {
    "qlora_18000_v2": ad.PREDS_QLORA[0], "jurado": ad.JURADO, "referencia_gpt": ad.REFERENCIA_GPT,
    "haiku_2700": ad.HAIKU_2700, "split_v2": ad.SPLIT_V2})
SHA_DATOS = sd.sha_archivo(EVIDENCIAS / "manifiesto_datos.json")
AUDITORIA = sd.auditar_fuentes(FUENTES)
(EVIDENCIAS / "auditoria_fuentes.json").write_text(json.dumps(AUDITORIA, ensure_ascii=False, indent=2), encoding="utf-8")

print("pool:", RES["info_pool"])
print("fugas:", RES["fugas"]["total_fugas"], "| citas de anios excluidos:", RES["fugas"]["citas_de_anios_excluidos"])
for n, c in MANIFIESTO["conteos"].items():
    print(f"{n:11s} n={MANIFIESTO['n'][n]:6d}", list(c.values()))

with rm.abrir_corrida("auditoria-datos", {"fase": 0, "sha_datos": SHA_DATOS, "dry": DRY}, {"manifiesto_datos_sha": SHA_DATOS}) as run:
    if run is not None:
        mlflow.log_metric("fugas_total", RES["fugas"]["total_fugas"])
        mlflow.log_metrics({f"n_{n}": v for n, v in MANIFIESTO["n"].items()})
        mlflow.log_dict(MANIFIESTO, "manifiesto_datos.json")
        mlflow.log_dict(AUDITORIA, "auditoria_fuentes.json")
        for par, v in AUDITORIA["acuerdo_pares"].items():
            if v["acuerdo"] is not None:
                mlflow.log_metric("acuerdo_" + par.replace("~", "_vs_"), v["acuerdo"])
AUDITORIA["acuerdo_pares"]

In [ ]:
AUDITORIA["precision_qlora_por_p"]

## Backfill de las 18 corridas de #103 (sin reentrenar)

In [ ]:
if not DRY:
    A4 = REPO / "proyecto_de_grado/artifacts/clasificacion_a4/corridas"
    print(len(rm.registrar_corridas_a4(A4, rm.EXPERIMENTO)), "runs de #103 nuevos")

## Fase 1 — Línea base TF-IDF + LR sobre D1

In [ ]:
COMUN = {"v_sha": MANIFIESTO["sha256_ids"]["validacion"], "datos_sha": SHA_DATOS}
D1 = VARIANTES["qlora_p50"]
correr("tfidf-lr-d1", ex.linea_base_tfidf, {**COMUN, "variante": "qlora_p50"}, D1, V, FUENTES, PAPERS, dry=DRY)

## Fase 2 — SciBERT-A (semilla 42) en D1…D6

In [ ]:
def sci(nombre, variante, semilla=42, modo="duro", pesos_clase=True, encoder="scibert"):
    cfg = {**COMUN, "variante": variante, "ids_train_sha": MANIFIESTO["sha256_ids"][variante]}
    hp = ct.Hiperparametros(pesos_clase=pesos_clase)
    return correr(nombre, ex.entrenar_y_registrar, cfg, VARIANTES[variante], V, FUENTES, PAPERS, CORRIDAS,
                  encoder=encoder, semilla=semilla, modo=modo, hp=hp, dry=DRY, manifiesto_datos_sha=SHA_DATOS)


FASE2 = {v: sci(f"scibert-A-{v}-s42", v) for v in sd.VARIANTES}
FASE2_PUNTAJE = {v: puntaje(r) for v, r in FASE2.items()}
MEJOR_DATOS = max(FASE2_PUNTAJE, key=FASE2_PUNTAJE.get)
print(FASE2_PUNTAJE, "-> mejor variante de datos:", MEJOR_DATOS)

## Fase 3 — Sobre la mejor variante: pérdida suave (KD), ponderada por p, sin pesos de clase

In [ ]:
FASE3 = {"duro": FASE2[MEJOR_DATOS],
         "kd": sci(f"scibert-A-{MEJOR_DATOS}-kd-s42", MEJOR_DATOS, modo="kd"),
         "ponderado": sci(f"scibert-A-{MEJOR_DATOS}-ponderado-s42", MEJOR_DATOS, modo="ponderado"),
         "sin_pesos": sci(f"scibert-A-{MEJOR_DATOS}-sinpesos-s42", MEJOR_DATOS, pesos_clase=False)}
FASE3_PUNTAJE = {k: puntaje(r) for k, r in FASE3.items()}
MEJOR_CONFIG = max(FASE3_PUNTAJE, key=FASE3_PUNTAJE.get)
CFG = {"duro": {}, "kd": {"modo": "kd"}, "ponderado": {"modo": "ponderado"},
       "sin_pesos": {"pesos_clase": False}}[MEJOR_CONFIG]
SUF = {"duro": "", "kd": "-kd", "ponderado": "-ponderado", "sin_pesos": "-sinpesos"}[MEJOR_CONFIG]
print(FASE3_PUNTAJE, "-> mejor configuracion:", MEJOR_CONFIG)

## Fase 4 — SPECTER-A con la mejor configuración

In [ ]:
R_SPECTER = sci(f"specter-A-{MEJOR_DATOS}{SUF}-s42", MEJOR_DATOS, encoder="specter", **CFG)

## Fase 5 — Mejor configuración de SciBERT con otras semillas

In [ ]:
R_SEMILLAS = {42: FASE3[MEJOR_CONFIG]}
for s in SEMILLAS[1:]:
    R_SEMILLAS[s] = sci(f"scibert-A-{MEJOR_DATOS}{SUF}-s{s}", MEJOR_DATOS, semilla=s, **CFG)
import statistics
for metrica in ("val_f1_macro", "dev_haiku_f1_macro", "ref_gpt_f1_macro"):
    xs = [mlflow.get_run(r).data.metrics.get(metrica) for r in R_SEMILLAS.values() if r]
    xs = [x for x in xs if x is not None]
    if len(xs) > 1:
        print(f"{metrica}: {statistics.mean(xs):.4f} +- {statistics.stdev(xs):.4f} (n={len(xs)})")

## Fase 6 — Smoke test de inferencia (SciBERT y SPECTER)

In [ ]:
SMOKE = []
for encoder, nombre in (("scibert", f"scibert-A-{MEJOR_DATOS}{SUF}-s42"), ("specter", f"specter-A-{MEJOR_DATOS}{SUF}-s42")):
    carpeta = CORRIDAS / nombre
    if not (carpeta / "model.safetensors").is_file():
        print("sin checkpoint:", nombre)
        continue
    try:
        r = ex.smoke_inferencia(carpeta, encoder, V, FUENTES, PAPERS)
    except Exception:
        FALLAS.append(f"smoke-{encoder}")
        r = {"encoder": encoder, "ok": False, "error": traceback.format_exc()}
    SMOKE.append(r)
    print(r)
(EVIDENCIAS / "smoke_inferencia.json").write_text(json.dumps(SMOKE, ensure_ascii=False, indent=2), encoding="utf-8")
with rm.abrir_corrida("smoke-inferencia", {"fase": 6, "mejor": f"{MEJOR_DATOS}{SUF}", "dry": DRY}) as run:
    if run is not None:
        for r in SMOKE:
            mlflow.log_metric(f"smoke_ok_{r['encoder']}", int(r["ok"]))
            if "latencia_ms" in r:
                mlflow.log_metric(f"smoke_latencia_ms_{r['encoder']}", r["latencia_ms"])

## Fase 7 — Entrada B (opcional)

Solo corre si existen las entradas B preparadas (`artifacts/experimentos_104/entrada_b/`). La preparación usa red y
GPU de día (ver runbook); de noche no se usa la red. Si no están, la fase queda registrada como omitida.

In [ ]:
ENTRADA_B = ARTIFACTS / "entrada_b"
with rm.abrir_corrida("fase7-entrada-b", {"fase": 7, "dry": DRY, "disponible": ENTRADA_B.is_dir()}) as run:
    if run is not None:
        mlflow.set_tag("estado_fase", "omitida_sin_entradas_b" if not ENTRADA_B.is_dir() else "pendiente_implementar")
print("entrada B disponible:", ENTRADA_B.is_dir())

## Fase 8 — D7 `sonnet` (opcional)

Solo si existe `etiquetas_sonnet.csv` (anotación con Sonnet, sección 6 del plan). Esta noche se omite.

In [ ]:
SONNET = REPO / "proyecto_de_grado/artifacts/anotacion_sonnet/etiquetas_sonnet.csv"
with rm.abrir_corrida("fase8-d7-sonnet", {"fase": 8, "dry": DRY, "disponible": SONNET.is_file()}) as run:
    if run is not None:
        mlflow.set_tag("estado_fase", "omitida_sin_etiquetas_sonnet" if not SONNET.is_file() else "pendiente_implementar")
print("etiquetas Sonnet disponibles:", SONNET.is_file())

## Fase 9 — Manifiesto de experimentos

In [ ]:
DF = rm.exportar_manifiesto(EVIDENCIAS, rm.EXPERIMENTO + SUFIJO)
print(len(DF), "runs exportados a", EVIDENCIAS)
print("fallas:", FALLAS or "ninguna", f"| {(time.time() - T0) / 3600:.2f} h")
cols = [c for c in ("tags.mlflow.runName", "status", "metrics.val_f1_macro", "metrics.dev_haiku_f1_macro", "metrics.ref_gpt_f1_macro") if c in DF]
DF[cols].sort_values("metrics.val_f1_macro", ascending=False) if "metrics.val_f1_macro" in cols else DF[cols]